# 🌍 Company Universe - Consolidation Complète

Ce notebook génère le **Company Universe consolidé** en fusionnant toutes les sources de données disponibles.

**Sources de données :**
- ✅ **Key Market Data** : Données de marché depuis `extract_key_market_data.ipynb` (500 entreprises S&P 500 avec 10-K)
  - Données S&P 500 (Weight, Price)
  - Métriques financières (Market Cap, Revenue, Net Income, EPS, FCF)
  - Secteurs et industries (yfinance)
  - Ratios financiers calculés (P/E Ratio, etc.)
- ✅ **Data Points 10-K** : Extraction géographie, segments, supply chain (500 entreprises)

**Hypothèse :**
- Les deux sources contiennent **exactement les mêmes 500 entreprises** (S&P 500 avec 10-K)
- Chaque ticker présent dans Key Market Data est également présent dans Data Points 10-K

**Objectif :**
- Fusionner Key Market Data + Data Points 10-K par ticker
- Créer une structure unifiée pour les 500 entreprises


## Étape 1 : Imports et Configuration


In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import Dict, Optional, List
from collections import defaultdict
import os
from tqdm import tqdm

# Configuration des chemins
BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data' / 'generated'

KEY_MARKET_DATA_FILE = DATA_DIR / 'key_market_data' / 'all_market_data.json'
DATA_POINTS_DIR = DATA_DIR / 'extracted_data_points'
OUTPUT_DIR = DATA_DIR / 'company_universe'

# Créer le dossier de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie : {OUTPUT_DIR}")


## Étape 2 : Charger Key Market Data


In [ ]:
print("📊 Chargement des Key Market Data depuis all_market_data.json...")
print(f"   Fichier: {KEY_MARKET_DATA_FILE}")

if not KEY_MARKET_DATA_FILE.exists():
    print(f"⚠️ ERREUR: Fichier non trouvé!")
    print(f"   Exécutez d'abord extract_key_market_data.ipynb pour générer ce fichier")
    raise FileNotFoundError(f"Fichier manquant: {KEY_MARKET_DATA_FILE}")

with open(KEY_MARKET_DATA_FILE, 'r', encoding='utf-8') as f:
    key_market_data = json.load(f)

print(f"✅ {len(key_market_data)} entreprises chargées depuis Key Market Data")
print(f"   - Toutes sont dans le S&P 500 et ont un 10-K disponible")

# Statistiques des données
sector_count = sum(1 for v in key_market_data.values() if v.get('sector') and v.get('sector') != 'Unknown')
performance_count = sum(1 for v in key_market_data.values() if v.get('market_cap'))

print(f"\n📊 Détails:")
print(f"   - Avec secteur (yfinance): {sector_count}")
print(f"   - Avec métriques financières: {performance_count}")
print(f"   - Avec prix S&P 500: {sum(1 for v in key_market_data.values() if v.get('current_price'))}")

# Afficher un exemple
example_ticker = 'AAPL'
if example_ticker in key_market_data:
    print(f"\n📋 Exemple ({example_ticker}):")
    example_data = key_market_data[example_ticker]
    print(f"   - Company: {example_data.get('company_name')}")
    print(f"   - Sector: {example_data.get('sector')}")
    print(f"   - Industry: {example_data.get('industry')}")
    print(f"   - S&P 500 Weight: {example_data.get('sp500_weight')}")
    print(f"   - Market Cap: ${example_data.get('market_cap', 0)/1e12:.2f}T")
    print(f"   - Revenue: ${example_data.get('revenue', 0)/1e9:.2f}B")
    print(f"   - P/E Ratio: {example_data.get('pe_ratio')}")
else:
    print(f"\n⚠️ {example_ticker} non trouvé dans les données")


## Étape 3 : Charger Data Points 10-K
 

In [ ]:
print("📄 Chargement des Data Points 10-K...")

# Lister tous les fichiers data points
data_points_files = list(DATA_POINTS_DIR.glob('*_data_points.json'))
print(f"📁 {len(data_points_files)} fichiers trouvés")

# Charger tous les data points
data_points = {}
for file_path in tqdm(data_points_files, desc="Chargement"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            ticker = data.get('ticker', '').upper()
            if ticker:
                data_points[ticker] = data
    except Exception as e:
        print(f"⚠️ Erreur lors du chargement de {file_path.name}: {e}")

print(f"\n✅ {len(data_points)} entreprises avec Data Points 10-K")
print(f"   (Attendu: 500 entreprises - mêmes que Key Market Data)")

# Afficher un exemple
if example_ticker in data_points:
    print(f"\n📋 Exemple ({example_ticker}):")
    print(f"   - Type: {data_points[example_ticker].get('company_type', 'N/A')}")
    print(f"   - Segments: {len(data_points[example_ticker].get('segments', []))} segments")
    print(f"   - Pays: {len(data_points[example_ticker].get('geography', {}).get('countries', []))} pays")


## Étape 4 : Fusionner les Données par Entreprise


In [ ]:
def merge_company_data(
    ticker: str,
    market_data: Optional[Dict],
    data_points: Optional[Dict]
) -> Dict:
    """
    Fusionne les données entreprises (Market Data + Data Points)
    """
    company = {
        'ticker': ticker,
        'data_sources': {
            'has_market_data': market_data is not None,
            'has_data_points': data_points is not None
        },
        'market_data': market_data or {},
        'data_points': data_points or {}
    }
    
    # Extraire le nom de l'entreprise depuis n'importe quelle source
    company_name = (
        market_data.get('company_name') if market_data
        else data_points.get('company_name') if data_points
        else None
    )
    
    if company_name:
        company['company_name'] = company_name
    
    return company

# Obtenir tous les tickers uniques
# BASE: Utiliser Key Market Data comme source principale (500 entreprises S&P 500 avec 10-K)
all_tickers = set(key_market_data.keys())

# Ajouter les tickers avec Data Points 10-K (certains peuvent ne pas être dans Key Market Data)
all_tickers.update(data_points.keys())

print(f"📊 Total de {len(all_tickers)} entreprises uniques à traiter")
print(f"   - Depuis Key Market Data: {len(key_market_data)}")
print(f"   - Avec Data Points 10-K: {len(data_points)}")
print(f"   - Intersection (les deux sources): {len(set(key_market_data.keys()) & set(data_points.keys()))}")
print(f"   - Uniquement dans Key Market Data: {len(set(key_market_data.keys()) - set(data_points.keys()))}")
print(f"   - Uniquement dans Data Points: {len(set(data_points.keys()) - set(key_market_data.keys()))}")


In [ ]:
def merge_company_data(
    ticker: str,
    market_data: Optional[Dict],
    data_points: Optional[Dict]
) -> Dict:
    """
    Fusionne les données entreprises (Market Data + Data Points)
    """
    company = {
        'ticker': ticker,
        'data_sources': {
            'has_market_data': market_data is not None,
            'has_data_points': data_points is not None
        },
        'market_data': market_data or {},
        'data_points': data_points or {}
    }
    
    # Extraire le nom de l'entreprise depuis n'importe quelle source
    company_name = (
        market_data.get('company_name') if market_data
        else data_points.get('company_name') if data_points
        else None
    )
    
    if company_name:
        company['company_name'] = company_name
    
    return company

# Obtenir tous les tickers - les deux sources ont les mêmes 500 entreprises
# BASE: Utiliser Key Market Data comme source principale (500 entreprises S&P 500 avec 10-K)
all_tickers = list(key_market_data.keys())

# Vérifier que les deux sources ont les mêmes entreprises
tickers_market = set(key_market_data.keys())
tickers_data_points = set(data_points.keys())

# Tickers manquants dans chaque source
missing_in_dp = tickers_market - tickers_data_points
missing_in_market = tickers_data_points - tickers_market

print(f"📊 Analyse des sources de données:")
print(f"   - Key Market Data: {len(key_market_data)} entreprises")
print(f"   - Data Points 10-K: {len(data_points)} entreprises")
print(f"   - Intersection: {len(tickers_market & tickers_data_points)} entreprises")
print(f"")

if missing_in_dp:
    print(f"⚠️ {len(missing_in_dp)} entreprises dans Key Market Data mais PAS dans Data Points:")
    print(f"   {', '.join(list(missing_in_dp)[:10])}{'...' if len(missing_in_dp) > 10 else ''}")
else:
    print(f"✅ Toutes les entreprises de Key Market Data sont présentes dans Data Points")

if missing_in_market:
    print(f"⚠️ {len(missing_in_market)} entreprises dans Data Points mais PAS dans Key Market Data:")
    print(f"   {', '.join(list(missing_in_market)[:10])}{'...' if len(missing_in_market) > 10 else ''}")
else:
    print(f"✅ Toutes les entreprises de Data Points sont présentes dans Key Market Data")

print(f"")
print(f"📋 Total entreprises à traiter: {len(all_tickers)}")
print(f"   (Base: Key Market Data - 500 entreprises S&P 500 avec 10-K)")


## Étape 6 : Générer le Company Universe


In [ ]:
print("🌍 Génération du Company Universe...")
print(f"   Base: {len(all_tickers)} entreprises (les mêmes dans Key Market Data et Data Points 10-K)")

company_universe = {}
stats = {
    'total_companies': len(all_tickers),
    'with_market_data': 0,
    'with_data_points': 0,
    'with_market_and_dp': 0  # Market Data + Data Points
}

# Parcourir toutes les entreprises (les deux sources ont les mêmes tickers)
for ticker in tqdm(all_tickers, desc="Fusion"):
    # Les deux sources contiennent les mêmes entreprises, donc on peut accéder directement
    market_data = key_market_data.get(ticker)
    dp_data = data_points.get(ticker)
    
    # Mettre à jour les statistiques
    if market_data:
        stats['with_market_data'] += 1
    if dp_data:
        stats['with_data_points'] += 1
    if market_data and dp_data:
        stats['with_market_and_dp'] += 1
    
    # Fusionner les données (Market Data + Data Points)
    company_universe[ticker] = merge_company_data(ticker, market_data, dp_data)

print(f"\n✅ Company Universe généré avec {len(company_universe)} entreprises")
print(f"\n📊 Statistiques:")
print(f"   - Total entreprises: {stats['total_companies']}")
print(f"   - Avec Market Data: {stats['with_market_data']} ({stats['with_market_data']/stats['total_companies']*100:.1f}%)")
print(f"   - Avec Data Points 10-K: {stats['with_data_points']} ({stats['with_data_points']/stats['total_companies']*100:.1f}%)")
print(f"   - Avec Market Data + Data Points: {stats['with_market_and_dp']} ({stats['with_market_and_dp']/stats['total_companies']*100:.1f}%)")

# Avertissement si toutes les entreprises n'ont pas les deux sources
if stats['with_market_and_dp'] < stats['total_companies']:
    missing_both = stats['total_companies'] - stats['with_market_and_dp']
    print(f"\n⚠️ {missing_both} entreprises n'ont pas les deux sources de données")


## Étape 7 : Sauvegarder le Company Universe


In [ ]:
# Sauvegarder en JSON complet (AVANT enrichissement, sera mis à jour après si enrichissement activé)
output_file_json = OUTPUT_DIR / 'company_universe.json'
with open(output_file_json, 'w', encoding='utf-8') as f:
    json.dump(company_universe, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier JSON sauvegardé: {output_file_json}")
if output_file_json.exists():
    file_size_mb = output_file_json.stat().st_size / 1024 / 1024
    print(f"   Taille: {file_size_mb:.2f} MB")
    print(f"   Entreprises: {len(company_universe)}")
    print("💡 Note: Si enrichissement activé, ce fichier sera mis à jour après")

# Sauvegarder aussi un résumé en CSV pour analyse rapide
summary_rows = []
for ticker, company in company_universe.items():
    row = {
        'ticker': ticker,
        'company_name': company.get('company_name', ''),
        'has_market_data': company['data_sources']['has_market_data'],
        'has_data_points': company['data_sources']['has_data_points']
    }
    
    # Ajouter quelques métriques clés si disponibles
    market = company.get('market_data', {})
    if market:
        row['market_cap'] = market.get('market_cap')
        row['sp500_weight'] = market.get('sp500_weight')
        row['current_price'] = market.get('current_price')
        row['sector'] = market.get('sector')
        row['industry'] = market.get('industry')
        row['revenue'] = market.get('revenue')
        row['net_income'] = market.get('net_income')
        row['pe_ratio'] = market.get('pe_ratio')
        row['profit_margin'] = market.get('profit_margin')
    
    dp = company.get('data_points', {})
    if dp:
        row['company_type'] = dp.get('company_type', '')
        geo = dp.get('geography', {})
        row['num_countries'] = len(geo.get('countries', []))
        row['has_eu'] = geo.get('has_eu', False)
        row['has_na'] = geo.get('has_na', False)
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
output_file_csv = OUTPUT_DIR / 'company_universe_summary.csv'
summary_df.to_csv(output_file_csv, index=False, encoding='utf-8')

print(f"✅ Fichier CSV résumé sauvegardé: {output_file_csv}")
print(f"\n📋 Aperçu du résumé:")
print(summary_df.head(10).to_string())


## Étape 8 : Enrichissement avec Yahoo Finance (Optionnel)

Enrichissement avec données temps réel pour améliorer précision des analyses d'impact.


In [ ]:
# Importer la fonction d'enrichissement
import sys
sys.path.append(str(BASE_DIR / 'scripts'))
try:
    from enrich_with_yfinance import enrich_company_universe
    
    # Option : Enrichir seulement les top N entreprises (pour limiter appels API)
    # Mettre None pour désactiver l'enrichissement
    # Note: Les 500 entreprises ont déjà les secteurs depuis extract_key_market_data
    # Cet enrichissement ajoute les données temps réel (prix actuel, beta, etc.)
    ENRICH_TOP_N = None  # Changer ce nombre selon besoin (None = désactiver)
    
    if ENRICH_TOP_N:
        print(f"📊 Enrichissement avec Yahoo Finance des {ENRICH_TOP_N} premières entreprises...")
        print("⏱️  Cela peut prendre quelques minutes...")
        
        company_universe_enriched = enrich_company_universe(
            company_universe, 
            limit_top_n=ENRICH_TOP_N
        )
        
        # Remplacer le Company Universe avec version enrichie
        company_universe = company_universe_enriched
        
        print("✅ Enrichissement terminé!")
        print("\n💡 Les données enrichies sont dans: market_data.realtime")
        print("   - current_price: Prix actuel (vs csv_price du CSV)")
        print("   - beta: Volatilité (pour ajuster risque)")
        print("   - volatility_factor: Facteur d'ajustement du risque")
        
        # Afficher un exemple
        enriched_count = sum(1 for c in company_universe.values() 
                            if c.get('market_data', {}).get('realtime', {}).get('success', False))
        print(f"\n📊 {enriched_count} entreprises enrichies avec succès")
        
        # Ré-sauvegarder avec données enrichies
        print("\n💾 Sauvegarde du Company Universe enrichi...")
        with open(output_file_json, 'w', encoding='utf-8') as f:
            json.dump(company_universe, f, indent=2, ensure_ascii=False)
        print(f"✅ Fichier mis à jour: {output_file_json}")
        
        # Afficher un exemple d'enrichissement
        example_ticker = None
        for ticker, data in company_universe.items():
            if data.get('market_data', {}).get('realtime', {}).get('success', False):
                example_ticker = ticker
                break
        
        if example_ticker:
            example = company_universe[example_ticker]['market_data']['realtime']
            print(f"\n📋 Exemple d'enrichissement ({example_ticker}):")
            print(f"   Prix CSV: ${example.get('csv_price', 0):.2f}")
            print(f"   Prix actuel: ${example.get('current_price', 0):.2f} ({example.get('price_change_pct', 0):+.2f}%)")
            print(f"   Beta: {example.get('beta', 1.0):.2f}")
            print(f"   Facteur volatilité: {example.get('volatility_factor', 1.0):.3f}")
        
    else:
        print("⚠️ Enrichissement désactivé (ENRICH_TOP_N = None)")
        print("💡 Pour activer: mettre ENRICH_TOP_N = 50 (par exemple)")
        
except ImportError as e:
    print(f"⚠️ Import error: {e}")
    print("💡 Installer yfinance: pip install yfinance")
    print("⚠️ Enrichissement ignoré, continuant avec données CSV seulement")
except Exception as e:
    print(f"⚠️ Erreur lors de l'enrichissement: {e}")
    print("⚠️ Continuant avec données CSV seulement")


## Étape 8 : Visualiser les Données Disponibles (Optionnel)


In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Configuration
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (12, 8)
    
    # Créer un graphique de disponibilité des données
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Disponibilité par source
    ax1 = axes[0, 0]
    source_counts = [
        stats['with_market_data'],
        stats['with_data_points'],
        stats['with_market_and_dp']
    ]
    source_labels = ['Market Data', 'Data Points 10-K', 'Market + Data Points']
    ax1.bar(source_labels, source_counts, color=['#2ecc71', '#3498db', '#e74c3c'])
    ax1.set_title('Disponibilité des Sources de Données', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Nombre d\'entreprises')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Combinaisons de sources
    ax2 = axes[0, 1]
    combinations = []
    for ticker, company in company_universe.items():
        src = company['data_sources']
        combo = []
        if src['has_market_data']:
            combo.append('M')
        if src['has_data_points']:
            combo.append('D')
        combinations.append(''.join(combo) if combo else 'None')
    
    from collections import Counter
    combo_counts = Counter(combinations)
    if combo_counts:
        ax2.barh(list(combo_counts.keys()), list(combo_counts.values()), color='#9b59b6')
        ax2.set_title('Combinaisons de Sources', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Nombre d\'entreprises')
        ax2.set_ylabel('Combinaison (M=Market, D=Data Points)')
    
    # 3. Distribution Market Cap (si disponible)
    ax3 = axes[1, 0]
    market_caps = [
        company.get('market_data', {}).get('market_cap')
        for company in company_universe.values()
        if company.get('market_data', {}).get('market_cap')
    ]
    if market_caps:
        ax3.hist([mc / 1e12 for mc in market_caps if mc], bins=30, color='#3498db', edgecolor='black')
        ax3.set_title('Distribution Market Cap (en billions USD)', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Market Cap (Billions USD)')
        ax3.set_ylabel('Nombre d\'entreprises')
        ax3.grid(axis='y', alpha=0.3)
    
    # 4. Présence géographique (EU vs NA)
    ax4 = axes[1, 1]
    eu_count = sum(
        1 for company in company_universe.values()
        if company.get('data_points', {}).get('geography', {}).get('has_eu', False)
    )
    na_count = sum(
        1 for company in company_universe.values()
        if company.get('data_points', {}).get('geography', {}).get('has_na', False)
    )
    geo_labels = ['Présence EU', 'Présence NA']
    geo_counts = [eu_count, na_count]
    ax4.bar(geo_labels, geo_counts, color=['#16a085', '#27ae60'])
    ax4.set_title('Présence Géographique (depuis 10-K)', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Nombre d\'entreprises')
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'company_universe_visualization.png', dpi=150, bbox_inches='tight')
    print(f"✅ Visualisation sauvegardée: {OUTPUT_DIR / 'company_universe_visualization.png'}")
    plt.show()
except ImportError:
    print("⚠️ matplotlib/seaborn non disponibles - visualisation ignorée")


## Étape 9 : Exemple d'Entreprise Complète


In [ ]:
# Afficher un exemple d'entreprise avec le plus de données disponibles
example_companies = []
for ticker, company in company_universe.items():
    src = company['data_sources']
    score = sum([src['has_market_data'], src['has_data_points']])
    if score >= 2:  # Au moins 2 sources (Market + Data Points)
        example_companies.append((ticker, company, score))

# Trier par score décroissant
example_companies.sort(key=lambda x: x[2], reverse=True)

if example_companies:
    best_ticker, best_company, score = example_companies[0]
    print(f"📊 Exemple d'entreprise avec données complètes: {best_ticker}")
    print(f"   Score de complétude: {score}/2 (Market Data + Data Points)")
    print(f"\n📋 Structure complète (premiers 2000 caractères):")
    print(json.dumps(best_company, indent=2, ensure_ascii=False)[:2000] + "...")
else:
    print("⚠️ Aucune entreprise avec au moins 2 sources de données")


## 📝 Notes et Prochaines Étapes

### ✅ Ce qui a été fait :
- Fusion Key Market Data + Data Points 10-K
- Statistiques et visualisations
- Structure prête pour jointures avec Legal Data (table séparée)

### 🔄 À faire :
- ✅ Créer Regulation table séparée pour Legal Data
- ✅ Implémenter jointures (ticker, secteur, pays) pour analyse d'impact
- ✅ Enrichir avec APIs externes (Yahoo Finance, SEC, Morningstar)
- ✅ Créer index de recherche (pour RAG)
- ✅ Optimiser pour analyse d'impact réglementaire
